In [ ]:
from pathlib import Path
import re

dataset_dir = Path("dataset")

folders = {
    "images": ".png",
    "depth": ".png",
    "info": ".txt",
    "masks": ".png",
    "pointcloud": ".ply",
    "labels": ".txt",
}
START_INDEX = 0

# Get indices from images folder
image_files = sorted((dataset_dir / "images").glob("img*.png"))

pattern = re.compile(r"img(\d+)")

indices = []
for f in image_files:
    m = pattern.search(f.stem)
    if m:
        indices.append(int(m.group(1)))

# Build mapping old_index -> new_index
mapping = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(sorted(indices), start=START_INDEX)
}

print(f"Shiffting {len(mapping)} files starting at {START_INDEX}")

# First pass: rename to temporary names to avoid collisions
for folder, ext in folders.items():
    folder_path = dataset_dir / folder

    for old_idx, new_idx in mapping.items():
        old_file = folder_path / f"img{old_idx:04d}{ext}"

        if old_file.exists():
            tmp_file = folder_path / f"tmp_{old_idx:04d}{ext}"
            old_file.rename(tmp_file)

# Second pass: rename temp -> final
for folder, ext in folders.items():
    folder_path = dataset_dir / folder

    for old_idx, new_idx in mapping.items():
        tmp_file = folder_path / f"tmp_{old_idx:04d}{ext}"

        if tmp_file.exists():
            new_file = folder_path / f"img{new_idx:04d}{ext}"
            tmp_file.rename(new_file)

print("Renumbering complete.")

Found 44 samples
Renumbering complete.


In [1]:
from pathlib import Path
import re
import shutil

src_dir = Path("dataset")
dst_dir = Path("dataset-466")

folders = {
    "images": ".png",
    "depth": ".png",
    "info": ".txt",
    "masks": ".png",
    "pointcloud": ".ply",
    "labels": ".txt",
}
START_INDEX = 466

# Create destination folders
for folder in folders:
    (dst_dir / folder).mkdir(parents=True, exist_ok=True)

# Get indices from images folder
image_files = sorted((src_dir / "images").glob("img*.png"))

pattern = re.compile(r"img(\d+)")

indices = []
for f in image_files:
    m = pattern.search(f.stem)
    if m:
        indices.append(int(m.group(1)))

# Build mapping old_index -> new_index
mapping = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(sorted(indices), start=START_INDEX)
}

print(f"Copying {len(mapping)} files to {dst_dir}")

# Copy + rename
for folder, ext in folders.items():
    for old_idx, new_idx in mapping.items():
        src_file = src_dir / folder / f"img{old_idx:04d}{ext}"
        dst_file = dst_dir / folder / f"img{new_idx:04d}{ext}"

        if src_file.exists():
            shutil.copy2(src_file, dst_file)

print("New dataset created safely.")

Copying 1101 files to dataset-466
New dataset created safely.


In [3]:
from pathlib import Path
import re

dataset_dir = Path("dataset-multiple-img-only-final/dataset")

folder = "images"
ext =".png"

START_INDEX = 1701

# Get indices from images folder
image_files = sorted((dataset_dir / folder).glob("img*.png"))

pattern = re.compile(r"img(\d+)")

indices = []
for f in image_files:
    m = pattern.search(f.stem)
    if m:
        indices.append(int(m.group(1)))

# Build mapping old_index -> new_index
mapping = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(sorted(indices), start=START_INDEX)
}

print(f"Shiffting {len(mapping)} files starting at {START_INDEX}")

# First pass: rename to temporary names to avoid collisions

# to temp
for old_idx in mapping:
    old_file = dataset_dir/ folder / f"img{old_idx:04d}{ext}"

    if old_file.exists():
        tmp_file = dataset_dir/ folder / f"tmp_{old_idx:04d}{ext}"
        old_file.rename(tmp_file)

# Second pass: rename temp -> final
for old_idx, new_idx in mapping.items():
    tmp_file = dataset_dir/ folder / f"tmp_{old_idx:04d}{ext}"

    if tmp_file.exists():
        new_file = dataset_dir/ folder / f"img{new_idx:04d}{ext}"
        tmp_file.rename(new_file)

print("Renumbering complete.")


Shiffting 110 files starting at 1701
Renumbering complete.


In [ ]:
import cv2
import re
from pathlib import Path

# Define paths (Adjust if your folders are named differently)
base_path = Path("dataset")
uncropped_dir = base_path / "images"
info_dir = base_path / "info"
output_dir = base_path / "fixed_cropped_images"  # New folder for safety
output_dir.mkdir(exist_ok=True)

def parse_crop_offsets(info_path):
    """Extracts top and left crop values from the info txt file."""
    text = info_path.read_text()
    # Looks for something like: left=250 right=520
    match = re.search(r"top=(\d+)\s+bottom=(\d+)\s+left=(\d+)\s+right=(\d+)", text)
    if match:
        return {
            "top": int(match.group(1)),
            "bottom": int(match.group(2)),
            "left": int(match.group(3)),
            "right": int(match.group(4))
        }
    return None

# Process all uncropped images
for img_path in uncropped_dir.glob("*.png"):
    img_name = img_path.stem
    info_path = info_dir / f"{img_name}.txt"
    
    if not info_path.exists():
        print(f"[SKIP] Missing info file for {img_name}")
        continue
        
    crop = parse_crop_offsets(info_path)
    if crop is None:
        print(f"[SKIP] Could not parse crop bounds for {img_name}")
        continue
        
    # Load original image
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    
    # Apply the exact same crop logic your main.py used
    top, bottom, left, right = crop["top"], crop["bottom"], crop["left"], crop["right"]
    
    # Determine slicing bounds
    y_end = h - bottom if bottom > 0 else h
    x_end = w - right if right > 0 else w
    
    cropped_img = img[top:y_end, left:x_end]
    
    # Save the new correctly cropped image
    cv2.imwrite(str(output_dir / f"{img_name}.png"), cropped_img)
    print(f"[FIXED] Cropped {img_name}.png to match mask dimensions.")

In [8]:
from pathlib import Path
import re

dataset_dir = Path("data")

folders = {
    "images": ".png",
    "depth": ".png",
    "info": ".txt",
    "masks": ".png",
    "pointcloud": ".ply",
    "labels": ".txt",
    "uncropped_rgb": ".png",
}

pattern = re.compile(r"img(\d+)")

folder_indices = {}

# Collect indices from each folder
for folder, ext in folders.items():
    folder_path = dataset_dir / folder

    indices = set()

    for file in folder_path.glob(f"*{ext}"):
        match = pattern.search(file.stem)
        if match:
            indices.add(int(match.group(1)))

    folder_indices[folder] = indices

    print(f"{folder:15} : {len(indices)} files")

# Union of all indices
all_indices = set.union(*folder_indices.values())

print("\n=== Missing Files Report ===")

for folder, indices in folder_indices.items():
    missing = sorted(all_indices - indices)

    if missing:
        print(f"\n{folder} is missing {len(missing)} index(es):")
        print(missing[:20])
        if len(missing) > 20:
            print("...")
    else:
        print(f"\n{folder}: OK")

images          : 2010 files
depth           : 2010 files
info            : 2010 files
masks           : 2010 files
pointcloud      : 2010 files
labels          : 2010 files
uncropped_rgb   : 2010 files

=== Missing Files Report ===

images: OK

depth: OK

info: OK

masks: OK

pointcloud: OK

labels: OK

uncropped_rgb: OK


In [ ]:
from pathlib import Path
import shutil

dataset_dir = Path("dataset")
backup_dir = dataset_dir / "_backup_removed"
backup_dir.mkdir(exist_ok=True)

files_to_move = {
    "depth": ".png",
    "info": ".txt",
    "masks": ".png",
    "pointcloud": ".ply",
    "uncropped_rgb": ".png",
}

idx = 976

for folder, ext in files_to_move.items():
    src = dataset_dir / folder / f"img{idx:04d}{ext}"

    if src.exists():
        dst = backup_dir / src.name
        shutil.move(str(src), str(dst))
        print(f"Moved {src} -> {dst}")

yolo segmentation split

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil

dataset_dir = Path("data")

images_dir = dataset_dir / "images"
labels_dir = dataset_dir / "labels"

output_dir = dataset_dir / "data_yolo_seg"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
RANDOM_STATE = 42

# --------------------------------------------------
# Collect valid image-label pairs
# --------------------------------------------------
samples = []

for img_path in sorted(images_dir.glob("*.png")):
    label_path = labels_dir / f"{img_path.stem}.txt"

    if label_path.exists():
        samples.append((img_path, label_path))

print(f"Found {len(samples)} samples")

# --------------------------------------------------
# Split
# --------------------------------------------------
train_samples, temp_samples = train_test_split(
    samples,
    test_size=(1 - TRAIN_RATIO),
    random_state=RANDOM_STATE,
    shuffle=True,
)

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_STATE,
)

splits = {
    "train": train_samples,
    "val": val_samples,
    "test": test_samples,
}

# --------------------------------------------------
# Create folders
# --------------------------------------------------
for split in splits:
    (output_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / split / "labels").mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Copy files
# --------------------------------------------------
for split, data in splits.items():

    for img_path, label_path in data:

        shutil.copy2(
            img_path,
            output_dir / split / "images" / img_path.name,
        )

        shutil.copy2(
            label_path,
            output_dir / split / "labels" / label_path.name,
        )

print("Segmentation split completed.")

yolo classification split

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil

dataset_dir = Path("data")

images_dir = dataset_dir / "images"
labels_dir = dataset_dir / "labels"

output_dir = dataset_dir / "data_yolo_cls"

RANDOM_STATE = 42

CLASS_NAMES = {
    "0": "copper",
    "1": "steel",
}

# --------------------------------------------------
# Collect samples
# --------------------------------------------------
samples = []

for img_path in sorted(images_dir.glob("*.png")):

    label_path = labels_dir / f"{img_path.stem}.txt"

    if not label_path.exists():
        continue

    with open(label_path, "r") as f:
        first_line = f.readline().strip()

    if not first_line:
        continue

    class_id = first_line.split()[0]

    if class_id not in CLASS_NAMES:
        print(f"Unknown class in {label_path}")
        continue

    samples.append((img_path, class_id))

print(f"Found {len(samples)} samples")

# --------------------------------------------------
# Stratified split
# --------------------------------------------------
labels = [class_id for _, class_id in samples]

train, temp = train_test_split(
    samples,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels,
)

temp_labels = [class_id for _, class_id in temp]

val, test = train_test_split(
    temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=temp_labels,
)

splits = {
    "train": train,
    "val": val,
    "test": test,
}

# --------------------------------------------------
# Create folders and copy images
# --------------------------------------------------
for split, data in splits.items():

    for img_path, class_id in data:

        class_name = CLASS_NAMES[class_id]

        dst_dir = output_dir / split / class_name
        dst_dir.mkdir(parents=True, exist_ok=True)

        shutil.copy2(
            img_path,
            dst_dir / img_path.name,
        )

# --------------------------------------------------
# Statistics
# --------------------------------------------------
print("\nDataset statistics:")

for split, data in splits.items():

    copper = sum(1 for _, c in data if c == "0")
    steel = sum(1 for _, c in data if c == "1")

    print(
        f"{split:5} | "
        f"total={len(data):4d} "
        f"copper={copper:4d} "
        f"steel={steel:4d}"
    )

print("\nYOLO Classification split completed.")